# VDJBet – Yellow Fever vaccination analysis

Replicates the `vdjbet_snippet.Rmd` analysis in Python:

1. Load LLWNGPMAV / HLA-A\*02 TRB entries from VDJdb
2. Download all YFV donor samples and load them with **Polars**
3. Build a `PgenGeneUsageAdjustment` from the combined samples to correct V/J bias
4. Create a `VDJBetOverlapAnalysis` with `n_mocks=100` and the pgen adjustment
5. For each sample × match mode, count overlapping clonotypes and cells
6. Z-test against the mock distribution; plot time courses and a heatmap

**Match modes** (`allow_1mm` × `match_v`/`match_j`):

| label | allow\_1mm | match\_v | match\_j |
|-------|-----------|---------|----------|
| exact\_vj   | False | True  | True  |
| 1mm\_vj     | True  | True  | True  |
| exact\_novj | False | False | False |
| 1mm\_novj   | True  | False | False |

In [ ]:
import math
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl

from mir.basic.gene_usage import GeneUsage
from mir.basic.pgen import PgenGeneUsageAdjustment
from mir.biomarkers.vdjbet import VDJBetOverlapAnalysis
from mir.common.parser import ClonotypeTableParser, load_vdjdb_latest
from mir.common.repertoire import LocusRepertoire, Repertoire

YFV_DIR = Path("assets/large/yfv19")

## 1  Load VDJdb reference (LLWNGPMAV, HLA-A\*02, TRB)

In [ ]:
vdjdb_rep = load_vdjdb_latest(
    epitope="LLWNGPMAV",
    locus="TRB",
    species="HomoSapiens",
    mhc_a_contains="A*02",
)
print(f"Reference: {vdjdb_rep.clonotype_count} unique clonotypes")
print(f"Example: {vdjdb_rep.clonotypes[0].junction_aa}  {vdjdb_rep.clonotypes[0].v_gene}")

## 2  Download YFV dataset from HuggingFace

The dataset is cached locally in `assets/large/yfv19`; the download runs once.

In [ ]:
if not YFV_DIR.exists() or not any(YFV_DIR.glob("*.airr.tsv.gz")):
    from huggingface_hub import snapshot_download
    print("Downloading isalgo/airr_yfv19 from HuggingFace ...")
    snapshot_download(
        repo_id="isalgo/airr_yfv19",
        repo_type="dataset",
        local_dir=str(YFV_DIR),
    )
    print(f"Downloaded to {YFV_DIR}")
else:
    print(f"Dataset already cached in {YFV_DIR}")

meta = pl.read_csv(YFV_DIR / "metadata.txt", separator="\t")
print(f"{len(meta)} samples:")
meta

## 3  Load YFV samples

Each AIRR TSV is loaded with **Polars** for speed, filtered to TRB clonotypes,
and parsed into a `LocusRepertoire`.  All samples are collected so we can build
a combined gene-usage profile for the pgen adjustment.

In [ ]:
_parser = ClonotypeTableParser()

samples: list[dict] = []
for row in meta.iter_rows(named=True):
    path = YFV_DIR / row["file_name"]
    df = pl.read_csv(path, separator="\t", infer_schema_length=10_000)
    if "locus" in df.columns:
        df = df.filter(pl.col("locus").fill_null("") == "TRB")
    df = df.filter(pl.col("junction_aa").is_not_null())
    df = df.filter(pl.col("junction_aa").str.strip_chars().str.len_chars() > 0)
    clones = _parser.parse_inner(df.to_pandas())
    rep = LocusRepertoire(
        clonotypes=clones, locus="TRB", repertoire_id=row["file_name"]
    )
    samples.append({
        "file_name": row["file_name"],
        "donor":     row["donor"],
        "day":       int(row["day"]),
        "replica":   str(row.get("replica", "1")),
        "repertoire": rep,
    })

print(f"Loaded {len(samples)} TRB samples")
print(
    f"Example: {samples[0]['donor']} day {samples[0]['day']}  "
    f"{samples[0]['repertoire'].clonotype_count:,} clonotypes"
)

## 4  Build pgen adjustment and create analysis object

A `PgenGeneUsageAdjustment` is built from the **combined** V/J gene usage of all
YFV samples.  Passing it to `VDJBetOverlapAnalysis` re-weights each OLGA-generated
sequence's Pgen by its V-J factor (target usage / OLGA usage), so the mock null
matches the actual protocol's V/J distribution without explicit V/J stratification
of the Pgen histogram.  This eliminates the false-positive enrichment that appears
at day 0 without adjustment.

In [ ]:
# Combined gene usage from all samples
all_clones = [c for s in samples for c in s["repertoire"].clonotypes]
combined_rep = Repertoire(clonotypes=all_clones, locus="TRB")
yfv_gu = GeneUsage.from_repertoire(combined_rep)

pgen_adj = PgenGeneUsageAdjustment(yfv_gu, seed=42)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    analysis = VDJBetOverlapAnalysis(
        vdjdb_rep,
        n_mocks=100,
        seed=42,
        pgen_adjustment=pgen_adj,
    )

print("VDJBetOverlapAnalysis created.")
print("Mock key sets are built lazily on the first score() call.")

## 5  Compute overlaps

`score()` is called for each (sample, match-mode) combination.  The analysis
object builds the mock null once and caches it; subsequent calls with different
`allow_1mm` / `match_v` / `match_j` values reuse the same mock key sets.

Each `OverlapResult` stores `n`, `dc`, `z_n`, `p_n`, `z_dc`, `p_dc`,
`allow_1mm`, `match_v`, `match_j`, plus the raw per-mock arrays.

In [ ]:
MODES = [
    dict(label="exact_vj",   allow_1mm=False, match_v=True,  match_j=True),
    dict(label="1mm_vj",     allow_1mm=True,  match_v=True,  match_j=True),
    dict(label="exact_novj", allow_1mm=False, match_v=False, match_j=False),
    dict(label="1mm_novj",   allow_1mm=True,  match_v=False, match_j=False),
]

rows: list[dict] = []
for si, s in enumerate(samples):
    rep = s["repertoire"]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        results = {
            m["label"]: analysis.score(
                rep,
                allow_1mm=m["allow_1mm"],
                match_v=m["match_v"],
                match_j=m["match_j"],
            )
            for m in MODES
        }

    r0 = results["exact_vj"]
    base = {
        "donor":   s["donor"],
        "day":     s["day"],
        "replica": s["replica"],
        "n_total":  r0.n_total,
        "dc_total": r0.dc_total,
    }
    for m in MODES:
        lbl = m["label"]
        r = results[lbl]
        base.update({
            f"n_{lbl}":      r.n,
            f"dc_{lbl}":     r.dc,
            f"frac_n_{lbl}": r.frac_n,
            f"frac_dc_{lbl}": r.frac_dc,
            f"z_n_{lbl}":    r.z_n,
            f"p_n_{lbl}":    r.p_n,
            f"z_dc_{lbl}":   r.z_dc,
            f"p_dc_{lbl}":   r.p_dc,
            f"mock_n_{lbl}": r.mock_n,
            f"dc_{lbl}_log2": math.log2(r.dc + 1),
            f"mock_dc_{lbl}_log2": [math.log2(x + 1) for x in r.mock_dc],
        })
    rows.append(base)
    if (si + 1) % 10 == 0:
        print(f"  {si + 1}/{len(samples)} samples processed")

df_res = pd.DataFrame(rows)
df_res[["donor", "day", "replica", "n_total",
        "n_exact_vj", "n_1mm_vj",
        "n_exact_novj", "n_1mm_novj"]].to_string(index=False)

## 6  Significance stars

Z-scores and p-values are already in `df_res`.  We add human-readable star
columns for each match mode.

In [ ]:
def _stars(p) -> str:
    if p is None:  return ""
    if p < 0.001:  return "***"
    if p < 0.01:   return "**"
    if p < 0.05:   return "*"
    return ""


for m in MODES:
    lbl = m["label"]
    df_res[f"stars_{lbl}"] = df_res[f"p_n_{lbl}"].apply(_stars)

df_res[["donor", "day", "replica",
        "z_n_exact_vj", "stars_exact_vj",
        "z_n_1mm_vj",   "stars_1mm_vj",
        "z_n_exact_novj", "stars_exact_novj",
        "z_n_1mm_novj",   "stars_1mm_novj"]]

## 7  Time-course plots

Four panels: `{exact, 1mm}` × `{match V+J, CDR3-only}`.  Red line = real overlap;
boxplots = mock null distribution.
Stars mark p < 0.05.

In [ ]:
donors   = sorted(df_res["donor"].unique())
replicas = sorted(df_res["replica"].unique())
days_all = sorted(df_res["day"].unique())


def _box_width(days: np.ndarray) -> float:
    if len(days) < 2:
        return 3.0
    gaps = np.diff(np.sort(days))
    return float(gaps[gaps > 0].min()) * 0.4


def _plot_timecourse(
    real_col: str, mock_col: str, stars_col: str,
    ylabel: str, title: str,
) -> None:
    fig, axes = plt.subplots(
        len(replicas), len(donors),
        figsize=(4 * len(donors), 3.5 * len(replicas)),
        squeeze=False,
        sharey=False,
    )
    fig.suptitle(title, fontsize=13, y=1.01)

    for ri, replica in enumerate(replicas):
        for di, donor in enumerate(donors):
            ax = axes[ri][di]
            sub = df_res[
                (df_res["donor"] == donor) & (df_res["replica"] == replica)
            ].sort_values("day")

            if sub.empty:
                ax.set_visible(False)
                continue

            days      = sub["day"].values
            real      = sub[real_col].values
            mock_data = list(sub[mock_col])
            width     = _box_width(days)

            ax.boxplot(
                mock_data,
                positions=days,
                widths=width,
                sym="k.",
                patch_artist=True,
                medianprops=dict(color="#888888", linewidth=1.5),
                boxprops=dict(facecolor="#d0e4f7", alpha=0.8),
                flierprops=dict(markersize=2, markerfacecolor="#888888"),
                manage_ticks=False,
            )

            ax.plot(days, real, "-o", color="#d62728",
                    linewidth=2, markersize=6, zorder=5)

            for day, rv, star in zip(days, real, sub[stars_col]):
                if star:
                    ax.text(day, rv + width * 0.1, star,
                            ha="center", va="bottom",
                            fontsize=11, color="#d62728", fontweight="bold")

            ax.set_title(f"{donor}  R{replica}", fontsize=9)
            ax.set_xlabel("Day post-vaccination", fontsize=8)
            ax.set_ylabel(ylabel, fontsize=8)
            ax.set_xticks(days_all)
            ax.tick_params(labelsize=8)

    plt.tight_layout()
    plt.show()


_plot_timecourse(
    real_col="n_exact_vj",
    mock_col="mock_n_exact_vj",
    stars_col="stars_exact_vj",
    ylabel="Matched clonotypes",
    title="LLWNGPMAV – exact match, V+J required",
)
_plot_timecourse(
    real_col="n_1mm_vj",
    mock_col="mock_n_1mm_vj",
    stars_col="stars_1mm_vj",
    ylabel="Matched clonotypes (1mm)",
    title="LLWNGPMAV – 1mm match, V+J required",
)
_plot_timecourse(
    real_col="n_exact_novj",
    mock_col="mock_n_exact_novj",
    stars_col="stars_exact_novj",
    ylabel="Matched clonotypes",
    title="LLWNGPMAV – exact match, CDR3-only",
)
_plot_timecourse(
    real_col="n_1mm_novj",
    mock_col="mock_n_1mm_novj",
    stars_col="stars_1mm_novj",
    ylabel="Matched clonotypes (1mm)",
    title="LLWNGPMAV – 1mm match, CDR3-only",
)

## 8  Heatmap: z-score and significance

Color scale is set automatically from the data range to capture the full
dynamic range.  Black borders mark cells with p < 0.05 and z > 0.

In [ ]:
def _heatmap(
    z_col: str, p_col: str, title: str,
    cmap: str = "RdBu_r",
    clim: tuple | None = None,
) -> None:
    df_res["sample_label"] = df_res["donor"] + "  R" + df_res["replica"]
    pivot_z = df_res.pivot_table(
        index="sample_label", columns="day", values=z_col, aggfunc="first"
    )
    pivot_p = df_res.pivot_table(
        index="sample_label", columns="day", values=p_col, aggfunc="first"
    )

    z_vals = pivot_z.values.astype(float)
    if clim is None:
        finite = z_vals[np.isfinite(z_vals)]
        if len(finite):
            abs_max = np.percentile(np.abs(finite), 98)
            clim = (-abs_max, abs_max)
        else:
            clim = (-4, 4)

    fig, ax = plt.subplots(figsize=(8, max(4, 0.6 * len(pivot_z))))
    im = ax.imshow(
        z_vals,
        aspect="auto", cmap=cmap,
        vmin=clim[0], vmax=clim[1],
        interpolation="nearest",
    )
    plt.colorbar(im, ax=ax, label="z-score", shrink=0.8)

    for ri in range(pivot_z.shape[0]):
        for ci in range(pivot_z.shape[1]):
            p = pivot_p.values[ri, ci]
            z = pivot_z.values[ri, ci]
            if pd.notna(p) and float(p) < 0.05 and pd.notna(z) and float(z) > 0:
                ax.add_patch(plt.Rectangle(
                    (ci - 0.5, ri - 0.5), 1, 1,
                    fill=False, edgecolor="black", linewidth=2,
                ))

    ax.set_xticks(range(len(pivot_z.columns)))
    ax.set_xticklabels(
        [f"Day {d}" for d in pivot_z.columns], rotation=45, ha="right"
    )
    ax.set_yticks(range(len(pivot_z.index)))
    ax.set_yticklabels(pivot_z.index)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


_heatmap("z_n_exact_vj",   "p_n_exact_vj",
         "Clonotype overlap (exact, V+J required) – z-score")
_heatmap("z_n_1mm_vj",     "p_n_1mm_vj",
         "Clonotype overlap (1mm, V+J required) – z-score")
_heatmap("z_n_exact_novj", "p_n_exact_novj",
         "Clonotype overlap (exact, CDR3-only) – z-score")
_heatmap("z_n_1mm_novj",   "p_n_1mm_novj",
         "Clonotype overlap (1mm, CDR3-only) – z-score")
_heatmap("z_dc_exact_vj",  "p_dc_exact_vj",
         "Cells matched (exact, V+J, log\u2082) – z-score",
         cmap="Reds", clim=(0, None))

## 9  Summary table

In [ ]:
keep_cols = [
    "donor", "day", "replica", "n_total", "dc_total",
]
for m in MODES:
    lbl = m["label"]
    keep_cols += [
        f"n_{lbl}", f"dc_{lbl}",
        f"frac_n_{lbl}", f"frac_dc_{lbl}",
        f"z_n_{lbl}", f"p_n_{lbl}", f"stars_{lbl}",
        f"z_dc_{lbl}", f"p_dc_{lbl}",
    ]

summary = (
    df_res[keep_cols]
    .copy()
    .sort_values(["donor", "replica", "day"])
    .reset_index(drop=True)
)

for m in MODES:
    lbl = m["label"]
    for c in [f"frac_n_{lbl}", f"frac_dc_{lbl}"]:
        summary[c] = summary[c].map("{:.2e}".format)
    for c in [f"z_n_{lbl}", f"z_dc_{lbl}"]:
        summary[c] = summary[c].round(2)
    for c in [f"p_n_{lbl}", f"p_dc_{lbl}"]:
        summary[c] = summary[c].map("{:.3f}".format)

pd.set_option("display.max_rows", 60)
summary